# 05. Выбор ансамбля и итоговый submission
Подключите распакованный baseline и Outputs этапов 01-04. CPU достаточно. Сначала сравниваются три одиночных кандидата и попарные смеси с фиксированной сеткой весов. Для победителя проверяются адаптивная калибровка этапа 01 и согласование этапа 02 по отдельности и совместно. Сетка и порядок при равенстве заданы заранее. Победитель определяется на tuning, затем выводится локальный test. Для конкурсного прогноза используются компоненты после полного refit из 00, 02, 03 и 04.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path.cwd()

def locate(name):
    paths = sorted(INPUT_ROOT.rglob(name))
    if len(paths) != 1:
        raise ValueError(f"Ожидался один {name}. Найдено: {paths}")
    return paths[0]

baseline = locate("weekly_predictions.parquet").parent
control = pd.read_parquet(baseline / "weekly_predictions.parquet")
future_control = pd.read_parquet(baseline / "full_predictions.parquet")
run_id = control.run_id.unique().item()
keys = ["cutoff","horizon","route_id"]
def validate(panel, template, labeled):
    cols = keys + (["y","period"] if labeled else ["id"])
    a = template[cols].merge(panel[cols], how="outer", on=keys, validate="one_to_one", indicator=True)
    if not a._merge.eq("both").all() or any(not a[k+"_x"].eq(a[k+"_y"]).all() for k in cols[len(keys):]):
        raise ValueError("Не совпадают ключи, метки или периоды прогнозов")
    if not np.isfinite(panel.raw_prediction).all():
        raise ValueError("Неконечный raw_prediction")

validate(control,control,True)
validate(future_control,future_control,False)
if not control.groupby("cutoff").size().eq(8000).all() or control.cutoff.nunique()!=42 or len(future_control)!=8000:
    raise ValueError("Ожидалась полная панель baseline")
if not control.model_info_through.lt(control.cutoff).all():
    raise ValueError("Некорректные даты обучения контроля")
def metrics(frame, prediction):
    y = frame.y.to_numpy(dtype=np.float64)
    p = np.asarray(prediction, dtype=np.float64)
    if p.shape != y.shape or not np.isfinite(p).all() or not np.isfinite(y).all():
        raise ValueError('Некорректные факты или прогнозы')
    e = p - y
    daily = pd.DataFrame(dict(day=frame.cutoff.dt.normalize().to_numpy(), y=y,
                              absolute=np.abs(e), signed=e)).groupby('day').sum()
    if (daily.y <= 0).any():
        raise ValueError('Неопределённая метрика: нулевой объём дня')
    return dict(rows=len(frame), days=len(daily), WAPE=float(daily.absolute.sum()/daily.y.sum()),
                signed_bias=float(daily.signed.sum()/daily.y.sum()),
                score=float((daily.absolute.sum()+abs(daily.signed.sum()))/daily.y.sum()),
                daily_score=float(((daily.absolute+daily.signed.abs())/daily.y).mean()))

def atomic_parquet(frame, path):
    temporary = path.with_suffix('.partial.parquet')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)

from itertools import combinations
from collections import OrderedDict
cal = json.loads(locate("calibration_settings.json").read_text())
volume = json.loads(locate("volume_settings.json").read_text())
if cal["run_id"] != run_id or volume["run_id"] != run_id:
    raise ValueError("Этапы 01 и 02 должны использовать один baseline")
models = OrderedDict()
models["baseline"] = (control,future_control)
for label in ["03_windows_features","04_losses"]:
    selection_path = locate(label+"_selection.json")
    config = json.loads(selection_path.read_text())
    panel = pd.read_parquet(selection_path.parent/(label+"_history.parquet"))
    future = pd.read_parquet(selection_path.parent/(label+"_future.parquet"))
    if config["run_id"]!=run_id or panel.run_id.unique().item()!=run_id or future.run_id.unique().item()!=run_id:
        raise ValueError(f"Другой запуск: {label}")
    validate(panel,control,True)
    validate(future,future_control,False)
    if not panel.model_info_through.lt(panel.cutoff).all():
        raise ValueError(f"Нарушена временная граница {label}")
    models[label] = (panel,future)
hist = control[keys+["period","y"]].sort_values(keys).reset_index(drop=True)
future = future_control[keys+["id"]].sort_values(keys).reset_index(drop=True)
for name,(part,full) in models.items():
    aligned = part.sort_values(keys)
    hist["raw_"+name] = aligned.raw_prediction.to_numpy()
    hist["pred_"+name] = aligned.y_pred.to_numpy()
    future["raw_"+name] = full.sort_values(keys).raw_prediction.to_numpy()
    future["pred_"+name] = full.sort_values(keys).y_pred.to_numpy()
vol_weight = float(volume["weight"])
if vol_weight:
    for frame, filename in [(hist,"volume_history.parquet"),(future,"volume_future.parquet")]:
        totals = pd.read_parquet(locate(filename)).set_index(["cutoff","horizon"]).total
        idx = pd.MultiIndex.from_frame(frame[["cutoff","horizon"]])
        frame["total"] = totals.reindex(idx).to_numpy()
        used = frame.period.ne("warmup") if "period" in frame else np.ones(len(frame),dtype=bool)
        if not np.isfinite(frame.loc[used,"total"]).all():
            raise ValueError("Пропущен прогноз общего объёма")
all_days = pd.DatetimeIndex(hist.cutoff.unique()).sort_values()
if not all_days.equals(pd.date_range(all_days[0],periods=42,freq="D")) or future.cutoff.unique().item()!=all_days[-1]+pd.Timedelta(days=1):
    raise ValueError("Проверьте непрерывность календаря")

## Выбор по двум неделям подбора


In [ ]:
def mix(frame, weights, prefix):
    return sum(w*frame[prefix+name].to_numpy(dtype=float) for name,w in weights.items())

def correction(weights):
    raw = mix(hist,weights,"raw_")
    errors = (hist.y-raw).groupby(hist.cutoff).mean()
    n, strength = cal["selected"].split("d_x")
    grid = errors.reindex(all_days.append(pd.DatetimeIndex(future.cutoff.unique())))
    return float(strength)*grid.shift(1).rolling(int(n),min_periods=1).mean()

def predict(frame, weights, adaptive, reconcile):
    p = mix(frame,weights,"pred_")
    if adaptive:
        p = np.maximum(0,mix(frame,weights,"raw_")+frame.cutoff.map(correction(weights)).to_numpy())
    if reconcile:
        totals = pd.Series(p,index=frame.index).groupby([frame.cutoff,frame.horizon]).transform("sum")
        if totals.le(0).any():
            raise ValueError("Нулевой суммарный прогноз")
        p = p*(1+vol_weight*(frame.total.to_numpy()/totals.to_numpy()-1))
    if not np.isfinite(p).all() or (p<0).any():
        raise ValueError("Некорректный итоговый прогноз")
    return p

tuning = hist.loc[hist.period.eq("tuning")]
holdout = hist.loc[hist.period.eq("holdout")]
candidates = [{name:1.0} for name in models]
for left,right in combinations(models,2):
    for share in [0.25,0.5,0.75]:
        candidates.append({left:1-share,right:share})
def tuning_row(weights, adaptive, reconcile):
    p = predict(tuning,weights,adaptive,reconcile)
    weeks = tuning.cutoff.lt(all_days[-21])
    scores = [metrics(tuning.loc[m],p[m.to_numpy()])["daily_score"] for m in [weeks,~weeks]]
    return dict(weights=weights, adaptive=adaptive, reconcile=reconcile, week1=scores[0],week2=scores[1],
                **metrics(tuning,p))

rows = [tuning_row(weights, False, False) for weights in candidates]
best_mix = min(rows, key=lambda r:r["daily_score"])
for adaptive, reconcile in [(True,False),(False,True),(True,True)]:
    if (adaptive and cal["selected"]=="fixed") or (reconcile and vol_weight==0):
        continue
    rows.append(tuning_row(best_mix["weights"],adaptive,reconcile))
choice = min(rows,key=lambda r:r["daily_score"])
settings = {k:choice[k] for k in ["weights","adaptive","reconcile"]}
Path("ensemble_selection.json").write_text(json.dumps(dict(run_id=run_id,**settings),indent=2))
ranking = pd.DataFrame(rows).sort_values("daily_score",kind="stable")
display(ranking.head(12))
ranking.to_csv("ensemble_tuning.csv",index=False)

## Проверка за 18-31 октября и сборка финального файла


In [ ]:
records = []
for name, cfg in [("weekly",dict(weights={"baseline":1.0},adaptive=False,reconcile=False)),("ensemble",settings)]:
    for period,frame in [("tuning",tuning),("test",holdout)]:
        p = predict(frame,**cfg)
        weeks = sorted(frame.cutoff.unique())
        for group, mask in [("all",frame.cutoff.isin(weeks)),("week1",frame.cutoff.isin(weeks[:7])),
                            ("week2",frame.cutoff.isin(weeks[7:])),("Saturday",frame.cutoff.dt.dayofweek.eq(5))]:
            records.append(dict(model=name,period=period,group=group,**metrics(frame.loc[mask],p[mask.to_numpy()])))
report = pd.DataFrame(records)
display(report)
report.to_csv("ensemble_report.csv",index=False)
print("Независимая оценка test получена до полного refit компонентов. Период ранее просматривался.")
final_prediction = pd.DataFrame(dict(id=future.id,y_pred=predict(future,**settings)))
expected = pd.read_csv(baseline/"submission.csv",usecols=["id"])
submission = expected.merge(final_prediction,on="id",how="left",validate="one_to_one")
if len(submission)!=8000 or not np.isfinite(submission.y_pred).all() or submission.y_pred.lt(0).any():
    raise ValueError("Некорректный submission")
submission.to_csv("submission.csv",index=False)
print("submission.csv", submission.shape, settings)